Before you turn this problem in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel$\rightarrow$Restart) and then **run all cells** (in the menubar, select Cell$\rightarrow$Run All).

Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE", as well as your name and collaborators below:

In [ ]:
NAME = ""
COLLABORATORS = ""

---

# Assignment 1: Understanding Leveraged ETF Behavior - TQQQ vs QQQ Analysis

Created by Dr. Pengfei Zhao

Beijing Normal University-Hong Kong Baptist University

**© 2025 Investment Practice Course**



### Learning Objectives
By completing this assignment, you will:
1. Understand how leveraged ETFs behave under different market conditions
2. Learn about volatility decay and its mathematical foundation
3. Compare theoretical models with real market data
4. Analyze the relationship between volatility and leveraged ETF performance

### Background
TQQQ is a 3x leveraged ETF that seeks to provide 3x the daily return of the QQQ ETF (which tracks the Nasdaq-100). However, due to daily rebalancing, TQQQ experiences **volatility decay** - it tends to underperform the 3x expected return over longer periods, especially in volatile sideways markets.

### Mathematical Foundation
The theoretical relationship for leveraged ETF returns is:

$$E[R_{leveraged}] \approx L \times E[R_{underlying}] - \frac{(L^2-L)}{2} \times Var[R_{underlying}]$$

For TQQQ with L=3:
$$E[R_{TQQQ}] \approx 3 \times E[R_{QQQ}] - 3 \times Var[R_{QQQ}]$$

This shows that higher volatility reduces leveraged ETF performance!

## Setup and Data Preparation

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Import the data loader module
from data_loader import load_etf_data, get_data_summary, calculate_rolling_metrics

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("📚 Libraries imported successfully!")
print("🎯 Ready to analyze TQQQ vs QQQ relationship!")
print("📊 Data loader module imported - using pre-downloaded data!")

---
# Part 1: Simulation Analysis (40 points)

## Task 1.1: Set Up Simulation Parameters (5 points)

In [ ]:
# SOLUTION: Set up simulation parameters
trading_days = 252  # One year of trading days
initial_price_qqq = 100
initial_price_tqqq = 100
np.random.seed(42)  # For reproducibility

print(f"✅ Simulation setup complete!")
print(f"📊 Trading days: {trading_days}")
print(f"💵 Initial prices: QQQ=${initial_price_qqq}, TQQQ=${initial_price_tqqq}")

## Task 1.2: Generate Market Scenarios (15 points)

In [ ]:
# SOLUTION: Generate the three market scenarios
low_vol_returns = np.random.normal(0, 0.01, trading_days)  # 0% mean, 1% daily volatility
high_vol_returns = np.random.normal(0, 0.03, trading_days)  # 0% mean, 3% daily volatility
trend_returns = np.random.normal(0.0008, 0.02, trading_days)  # 0.08% daily drift, 2% volatility

print("📈 Market scenarios generated:")
print(f"Low vol mean: {low_vol_returns.mean():.4f}, std: {low_vol_returns.std():.4f}")
print(f"High vol mean: {high_vol_returns.mean():.4f}, std: {high_vol_returns.std():.4f}")
print(f"Trend mean: {trend_returns.mean():.4f}, std: {trend_returns.std():.4f}")

## Task 1.3: Create TQQQ Simulation Function (10 points)

In [ ]:
def simulate_leveraged_etf(underlying_returns, leverage=3):
    """
    SOLUTION: Simulate leveraged ETF performance with daily rebalancing
    """
    # 1. Calculate leveraged returns
    leveraged_returns = leverage * underlying_returns
    
    # 2. Calculate cumulative prices using compound growth
    underlying_prices = initial_price_qqq * np.cumprod(1 + underlying_returns)
    leveraged_prices = initial_price_tqqq * np.cumprod(1 + leveraged_returns)
    
    # 3. Return all four arrays
    return underlying_prices, leveraged_prices, underlying_returns, leveraged_returns

print("🔧 Simulation function created!")

## Task 1.4: Run Simulations and Create Visualizations (10 points)

In [ ]:
# SOLUTION: Generate scenarios using the function
low_vol_qqq, low_vol_tqqq, low_vol_rets_qqq, low_vol_rets_tqqq = simulate_leveraged_etf(low_vol_returns)
high_vol_qqq, high_vol_tqqq, high_vol_rets_qqq, high_vol_rets_tqqq = simulate_leveraged_etf(high_vol_returns)
trend_qqq, trend_tqqq, trend_rets_qqq, trend_rets_tqqq = simulate_leveraged_etf(trend_returns)

# Create comprehensive visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
days = range(trading_days)

# Plot 1: Low volatility scenario
ax1.plot(days, low_vol_qqq, 'b-', linewidth=2, label='QQQ (Low Vol)', alpha=0.8)
ax1.plot(days, low_vol_tqqq, 'r-', linewidth=2, label='TQQQ (3x Leveraged)', alpha=0.8)
ax1.set_title('Low Volatility Sideways Market\n(1% daily vol, 0% trend)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price ($)', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.axhline(y=100, color='black', linestyle='--', alpha=0.5)

# Plot 2: High volatility scenario
ax2.plot(days, high_vol_qqq, 'b-', linewidth=2, label='QQQ (High Vol)', alpha=0.8)
ax2.plot(days, high_vol_tqqq, 'r-', linewidth=2, label='TQQQ (3x Leveraged)', alpha=0.8)
ax2.set_title('High Volatility Sideways Market\n(3% daily vol, 0% trend)', fontsize=14, fontweight='bold')
ax2.set_ylabel('Price ($)', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.axhline(y=100, color='black', linestyle='--', alpha=0.5)

# Plot 3: Trending market scenario
ax3.plot(days, trend_qqq, 'b-', linewidth=2, label='QQQ (Trending)', alpha=0.8)
ax3.plot(days, trend_tqqq, 'r-', linewidth=2, label='TQQQ (3x Leveraged)', alpha=0.8)
ax3.set_title('Trending Market\n(2% daily vol, positive trend)', fontsize=14, fontweight='bold')
ax3.set_xlabel('Trading Days', fontsize=12)
ax3.set_ylabel('Price ($)', fontsize=12)
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.axhline(y=100, color='black', linestyle='--', alpha=0.5)

# Plot 4: Return comparison bar chart
scenarios = ['Low Vol\nSideways', 'High Vol\nSideways', 'Trending\nMarket']
qqq_returns = [
    (low_vol_qqq[-1] / initial_price_qqq - 1) * 100,
    (high_vol_qqq[-1] / initial_price_qqq - 1) * 100,
    (trend_qqq[-1] / initial_price_qqq - 1) * 100
]
tqqq_returns = [
    (low_vol_tqqq[-1] / initial_price_tqqq - 1) * 100,
    (high_vol_tqqq[-1] / initial_price_tqqq - 1) * 100,
    (trend_tqqq[-1] / initial_price_tqqq - 1) * 100
]
expected_tqqq = [r * 3 for r in qqq_returns]

x = np.arange(len(scenarios))
width = 0.25

bars1 = ax4.bar(x - width, qqq_returns, width, label='QQQ Actual', color='blue', alpha=0.7)
bars2 = ax4.bar(x, tqqq_returns, width, label='TQQQ Actual', color='red', alpha=0.7)
bars3 = ax4.bar(x + width, expected_tqqq, width, label='TQQQ Expected (3x)', color='orange', alpha=0.7)

ax4.set_title('Volatility Decay Effect', fontsize=14, fontweight='bold')
ax4.set_xlabel('Market Scenario', fontsize=12)
ax4.set_ylabel('Total Return (%)', fontsize=12)
ax4.set_xticks(x)
ax4.set_xticklabels(scenarios)
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')
ax4.axhline(y=0, color='black', linestyle='-', alpha=0.5)

# Add value labels
def add_labels(ax, bars):
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, height),
                   xytext=(0, 3 if height >= 0 else -15), textcoords="offset points",
                   ha='center', va='bottom' if height >= 0 else 'top', fontsize=10)

add_labels(ax4, bars1)
add_labels(ax4, bars2)
add_labels(ax4, bars3)

plt.tight_layout()
plt.show()

print("📊 Simulation analysis complete!")

---
# Part 2: Real Data Analysis (40 points)

## Task 2.1: Load Pre-Downloaded Real Data (10 points)

In [ ]:
# SOLUTION: Load the pre-downloaded data
data, returns = load_etf_data()

if data is not None and returns is not None:
    # Display data summary
    get_data_summary(data, returns)
    
    # Calculate cumulative returns for plotting
    cumulative_returns = (1 + returns).cumprod()
    
    print(f"✅ Data loaded successfully")
    print(f"📊 Ready for analysis")
else:
    print("❌ Failed to load data")

## Task 2.2: Calculate Rolling Volatility (10 points)

In [ ]:
# SOLUTION: Calculate rolling metrics
# Use the calculate_rolling_metrics function from data_loader

# Calculate rolling metrics using data loader function
rolling_metrics = calculate_rolling_metrics(returns, window=30)

# Extract pre-calculated rolling metrics
rolling_vol = rolling_metrics['QQQ_volatility']
rolling_corr = rolling_metrics['correlation']

print(f"📊 30-day rolling metrics calculated using data_loader")
print(f"📈 Average volatility: {rolling_vol.mean():.2%}")
print(f"📉 Volatility range: {rolling_vol.min():.2%} - {rolling_vol.max():.2%}")
print(f"🔗 Average QQQ-TQQQ correlation: {rolling_corr.mean():.3f}")
print(f"🔢 Available metrics: {', '.join(rolling_metrics.columns)}")

## Task 2.3: Apply Theoretical Model (10 points)

In [ ]:
# SOLUTION: Apply theoretical model

# Use pre-calculated rolling variance from data_loader
rolling_var_qqq = rolling_metrics['QQQ_variance']

# Apply theoretical formula: E[TQQQ] = 3*QQQ_returns - 3*Var[QQQ]
# Use actual daily QQQ returns (not rolling mean) + volatility decay adjustment
theoretical_tqqq_return = 3 * returns['QQQ'] - 3 * rolling_var_qqq

# Calculate simple 3x leverage for comparison (no volatility adjustment)
simple_3x_return = 3 * returns['QQQ']

# Calculate cumulative performance
actual_tqqq_cumret = cumulative_returns['TQQQ']
theoretical_tqqq_cumret = (1 + theoretical_tqqq_return).cumprod()
simple_3x_cumret = (1 + simple_3x_return).cumprod()

print("🧮 Theoretical model applied")
print(f"📊 Theoretical vs Actual TQQQ comparison ready")
print(f"📈 Theoretical TQQQ annual return: {theoretical_tqqq_return.mean()*252:.2%}")
print(f"📈 Actual TQQQ annual return: {returns['TQQQ'].mean()*252:.2%}")
print(f"📈 Simple 3x annual return: {simple_3x_return.mean()*252:.2%}")

## Task 2.4: Create Comparison Analysis (10 points)

In [ ]:
# SOLUTION: Create comparison visualizations and analyze defined market periods

# Define the four periods explicitly with specific dates
period1_start = pd.to_datetime('2024-02-15')  # Low volatility period
period1_end = pd.to_datetime('2024-05-15')

period2_start = pd.to_datetime('2024-07-15')  # High volatility sideways period  
period2_end = pd.to_datetime('2024-10-15')

period3_start = pd.to_datetime('2025-01-15')  # High volatility trending down period
period3_end = pd.to_datetime('2025-04-15')

period4_start = pd.to_datetime('2024-11-01')  # High volatility trending up period
period4_end = pd.to_datetime('2025-01-10')   # Changed to avoid date not in dataset

# Create 2x2 subplot: 4 different market periods
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(18, 12))

# Function to calculate period-specific cumulative returns
def get_period_cumulative_returns(start_date, end_date):
    """Calculate cumulative returns starting from the period's start date"""
    period_returns = returns.loc[start_date:end_date]
    period_theoretical = theoretical_tqqq_return.loc[start_date:end_date]
    period_simple3x = (3 * returns['QQQ']).loc[start_date:end_date]
    
    # Calculate cumulative returns starting from 1.0 for this specific period
    period_qqq_cumret = (1 + period_returns['QQQ']).cumprod()
    period_tqqq_cumret = (1 + period_returns['TQQQ']).cumprod()
    period_theo_cumret = (1 + period_theoretical).cumprod()
    period_3x_cumret = (1 + period_simple3x).cumprod()
    
    return period_qqq_cumret, period_tqqq_cumret, period_theo_cumret, period_3x_cumret

# Plot 1: Period 1 - Low Volatility
p1_qqq, p1_tqqq, p1_theo, p1_3x = get_period_cumulative_returns(period1_start, period1_end)

ax1.plot(p1_qqq.index, p1_qqq, 'b-', linewidth=2, label='QQQ', alpha=0.8)
ax1.plot(p1_tqqq.index, p1_tqqq, 'r-', linewidth=2, label='TQQQ Actual', alpha=0.8)
ax1.plot(p1_theo.index, p1_theo, 'g--', linewidth=2, label='TQQQ Theoretical', alpha=0.8)
ax1.plot(p1_3x.index, p1_3x, 'orange', linestyle=':', linewidth=2, label='Simple 3x QQQ', alpha=0.8)
ax1.set_title('🟢 Period 1: Low Volatility\n2024-02-15 to 2024-05-15', fontsize=12, fontweight='bold')
ax1.set_ylabel('Cumulative Return (from Period Start)', fontsize=10)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Plot 2: Period 2 - High Volatility Sideways
p2_qqq, p2_tqqq, p2_theo, p2_3x = get_period_cumulative_returns(period2_start, period2_end)

ax2.plot(p2_qqq.index, p2_qqq, 'b-', linewidth=2, label='QQQ', alpha=0.8)
ax2.plot(p2_tqqq.index, p2_tqqq, 'r-', linewidth=2, label='TQQQ Actual', alpha=0.8)
ax2.plot(p2_theo.index, p2_theo, 'g--', linewidth=2, label='TQQQ Theoretical', alpha=0.8)
ax2.plot(p2_3x.index, p2_3x, 'orange', linestyle=':', linewidth=2, label='Simple 3x QQQ', alpha=0.8)
ax2.set_title('🟡 Period 2: High Volatility Sideways\n2024-07-15 to 2024-10-15', fontsize=12, fontweight='bold')
ax2.set_ylabel('Cumulative Return (from Period Start)', fontsize=10)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# Plot 3: Period 3 - High Volatility Trending Down
p3_qqq, p3_tqqq, p3_theo, p3_3x = get_period_cumulative_returns(period3_start, period3_end)

ax3.plot(p3_qqq.index, p3_qqq, 'b-', linewidth=2, label='QQQ', alpha=0.8)
ax3.plot(p3_tqqq.index, p3_tqqq, 'r-', linewidth=2, label='TQQQ Actual', alpha=0.8)
ax3.plot(p3_theo.index, p3_theo, 'g--', linewidth=2, label='TQQQ Theoretical', alpha=0.8)
ax3.plot(p3_3x.index, p3_3x, 'orange', linestyle=':', linewidth=2, label='Simple 3x QQQ', alpha=0.8)
ax3.set_title('🔴 Period 3: High Volatility Trending Down\n2025-01-15 to 2025-04-15', fontsize=12, fontweight='bold')
ax3.set_ylabel('Cumulative Return (from Period Start)', fontsize=10)
ax3.set_xlabel('Date', fontsize=10)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

# Plot 4: Period 4 - High Volatility Trending Up
p4_qqq, p4_tqqq, p4_theo, p4_3x = get_period_cumulative_returns(period4_start, period4_end)

ax4.plot(p4_qqq.index, p4_qqq, 'b-', linewidth=2, label='QQQ', alpha=0.8)
ax4.plot(p4_tqqq.index, p4_tqqq, 'r-', linewidth=2, label='TQQQ Actual', alpha=0.8)
ax4.plot(p4_theo.index, p4_theo, 'g--', linewidth=2, label='TQQQ Theoretical', alpha=0.8)
ax4.plot(p4_3x.index, p4_3x, 'orange', linestyle=':', linewidth=2, label='Simple 3x QQQ', alpha=0.8)
ax4.set_title('🟣 Period 4: High Volatility Trending Up\n2024-11-01 to 2025-01-10', fontsize=12, fontweight='bold')
ax4.set_ylabel('Cumulative Return (from Period Start)', fontsize=10)
ax4.set_xlabel('Date', fontsize=10)
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# EXPLICITLY DEFINED MARKET PERIODS ANALYSIS
print("\n📊 DEFINED MARKET PERIODS FOR PART 3 ANALYSIS:")
print("=" * 70)

print(f"📋 THE FOUR ANALYSIS PERIODS:")
print(f"\n🟢 PERIOD 1 - LOW VOLATILITY PERIOD:")
print(f"   📅 Dates: {period1_start.strftime('%Y-%m-%d')} to {period1_end.strftime('%Y-%m-%d')}")

# Calculate metrics for Period 1
p1_vol = rolling_vol.loc[period1_start:period1_end].mean()
p1_qqq_return = (data['QQQ'].loc[period1_end] / data['QQQ'].loc[period1_start] - 1) * 100
p1_tqqq_return = (data['TQQQ'].loc[period1_end] / data['TQQQ'].loc[period1_start] - 1) * 100
p1_expected_3x = p1_qqq_return * 3
p1_decay = p1_tqqq_return - p1_expected_3x

print(f"   📊 Average Volatility: {p1_vol:.1%}")
print(f"   📈 QQQ Return: {p1_qqq_return:+.1f}%")
print(f"   📈 TQQQ Actual Return: {p1_tqqq_return:+.1f}%")
print(f"   📈 Expected 3x Return: {p1_expected_3x:+.1f}%")
print(f"   📉 Volatility Decay: {p1_decay:+.1f}%")

print(f"\n🟡 PERIOD 2 - HIGH VOLATILITY SIDEWAYS PERIOD:")
print(f"   📅 Dates: {period2_start.strftime('%Y-%m-%d')} to {period2_end.strftime('%Y-%m-%d')}")

# Calculate metrics for Period 2
p2_vol = rolling_vol.loc[period2_start:period2_end].mean()
p2_qqq_return = (data['QQQ'].loc[period2_end] / data['QQQ'].loc[period2_start] - 1) * 100
p2_tqqq_return = (data['TQQQ'].loc[period2_end] / data['TQQQ'].loc[period2_start] - 1) * 100
p2_expected_3x = p2_qqq_return * 3
p2_decay = p2_tqqq_return - p2_expected_3x

print(f"   📊 Average Volatility: {p2_vol:.1%}")
print(f"   📈 QQQ Return: {p2_qqq_return:+.1f}% (sideways movement)")
print(f"   📈 TQQQ Actual Return: {p2_tqqq_return:+.1f}%")
print(f"   📈 Expected 3x Return: {p2_expected_3x:+.1f}%")
print(f"   📉 Volatility Decay: {p2_decay:+.1f}%")

print(f"\n🔴 PERIOD 3 - HIGH VOLATILITY TRENDING DOWN PERIOD:")
print(f"   📅 Dates: {period3_start.strftime('%Y-%m-%d')} to {period3_end.strftime('%Y-%m-%d')}")

# Calculate metrics for Period 3
p3_vol = rolling_vol.loc[period3_start:period3_end].mean()
p3_qqq_return = (data['QQQ'].loc[period3_end] / data['QQQ'].loc[period3_start] - 1) * 100
p3_tqqq_return = (data['TQQQ'].loc[period3_end] / data['TQQQ'].loc[period3_start] - 1) * 100
p3_expected_3x = p3_qqq_return * 3
p3_decay = p3_tqqq_return - p3_expected_3x

print(f"   📊 Average Volatility: {p3_vol:.1%}")
print(f"   📈 QQQ Return: {p3_qqq_return:+.1f}% (downward trend)")
print(f"   📈 TQQQ Return: {p3_tqqq_return:+.1f}%")
print(f"   📈 Expected 3x Return: {p3_expected_3x:+.1f}%")
print(f"   📉 Volatility Decay: {p3_decay:+.1f}%")

print(f"\n🟣 PERIOD 4 - HIGH VOLATILITY TRENDING UP PERIOD:")
print(f"   📅 Dates: {period4_start.strftime('%Y-%m-%d')} to {period4_end.strftime('%Y-%m-%d')}")

# Calculate metrics for Period 4
p4_vol = rolling_vol.loc[period4_start:period4_end].mean()
p4_qqq_return = (data['QQQ'].loc[period4_end] / data['QQQ'].loc[period4_start] - 1) * 100
p4_tqqq_return = (data['TQQQ'].loc[period4_end] / data['TQQQ'].loc[period4_start] - 1) * 100
p4_expected_3x = p4_qqq_return * 3
p4_decay = p4_tqqq_return - p4_expected_3x

print(f"   📊 Average Volatility: {p4_vol:.1%}")
print(f"   📈 QQQ Return: {p4_qqq_return:+.1f}% (upward trend)")
print(f"   📈 TQQQ Return: {p4_tqqq_return:+.1f}%")
print(f"   📈 Expected 3x Return: {p4_expected_3x:+.1f}%")
print(f"   📉 Volatility Decay: {p4_decay:+.1f}%")

# Summary comparison
print(f"\n📊 OVERALL ANALYSIS SUMMARY:")
print("=" * 60)
final_qqq = cumulative_returns['QQQ'].iloc[-1]
final_tqqq_actual = actual_tqqq_cumret.iloc[-1]
final_tqqq_theoretical = theoretical_tqqq_cumret.iloc[-1]
final_3x_simple = simple_3x_cumret.iloc[-1]

print(f"\nCUMULATIVE PERFORMANCE (Full Period):")
print(f"QQQ Total Return: {(final_qqq-1)*100:+.1f}%")
print(f"TQQQ Actual: {(final_tqqq_actual-1)*100:+.1f}%")
print(f"TQQQ Theoretical: {(final_tqqq_theoretical-1)*100:+.1f}%")
print(f"Simple 3x QQQ: {(final_3x_simple-1)*100:+.1f}%")

print("\n📊 Compare these FOUR PERIODS to understand TQQQ behavior!")
print("🎯 Key Insight: Notice how volatility decay affects TQQQ differently in each scenario!")

---
# Part 3: Analysis and Questions - SOLUTIONS (20 points)

## Question 1 (10 points)
Based on your simulation results, explain why TQQQ underperforms in high volatility sideways markets. Use specific numbers from your analysis.

**SOLUTION:**

In [ ]:
# SOLUTION: Analysis for Question 1
print("ANSWER TO QUESTION 1:")
print("=" * 50)
print(f"""TQQQ underperforms in high volatility sideways markets due to volatility decay caused by daily rebalancing.

SPECIFIC EVIDENCE FROM OUR ANALYSIS:

1. SIMULATION RESULTS:
   - In the LOW volatility scenario (1% daily vol), TQQQ performed close to 3x QQQ
   - In the HIGH volatility scenario (3% daily vol), TQQQ lost significantly more than expected
   - This demonstrates the mathematical relationship: E[TQQQ] ≈ 3×E[QQQ] - 3×Var[QQQ]

2. REAL DATA - PERIOD 2 (HIGH VOLATILITY SIDEWAYS):
   - Period: 2024-07-15 to 2024-10-15 (explicitly defined in Part 2)
   - Average Volatility: {p2_vol:.1%} (high volatility)
   - QQQ Return: {p2_qqq_return:+.1f}% (sideways movement)
   - TQQQ Actual Return: {p2_tqqq_return:+.1f}%
   - Expected 3x Return: {p2_expected_3x:+.1f}%
   - Volatility Decay Cost: {p2_decay:+.1f}%

3. COMPARISON WITH PERIOD 1 (LOW VOLATILITY):
   - Period: 2024-02-15 to 2024-05-15
   - Average Volatility: {p1_vol:.1%} (low volatility)
   - QQQ Return: {p1_qqq_return:+.1f}%
   - TQQQ Actual Return: {p1_tqqq_return:+.1f}%
   - Volatility Decay: {p1_decay:+.1f}% (much smaller decay)

4. MATHEMATICAL EXPLANATION:
   The volatility decay penalty is proportional to the SQUARE of volatility:
   - Period 1 (low vol): Minimal decay penalty
   - Period 2 (high vol): Large decay penalty of {abs(p2_decay):.1f}%
   
   This demonstrates that higher volatility creates exponentially higher costs!

5. MECHANISM:
   Daily rebalancing forces TQQQ to buy high and sell low to maintain 3x leverage.
   In Period 2's volatile sideways market, this created systematic losses that compounded.
   Even though QQQ had minimal directional movement, TQQQ lost significant value due to 
   the path-dependent nature of leveraged returns.""")

## Question 2 (10 points)

Based on your findings, what trading strategies would you recommend for TQQQ? When should investors use it and when should they avoid it? 

In [ ]:
# SOLUTION: Analysis for Question 4
print("ANSWER TO QUESTION 4:")
print("=" * 50)
print("""COMPREHENSIVE TRADING STRATEGY FOR TQQQ:

✅ WHEN TO USE TQQQ:

1. **Strong Trending Markets**:
   - Clear technical breakouts with momentum
   - Bull market phases with sustained uptrends
   - Strong sector rotation into technology

2. **Low Volatility Environments**:
   - VIX below 20 and falling
   - Stable macro conditions
   - Clear Fed policy direction

3. **Short-term Tactical Plays**:
   - Day trading with tight stops
   - Swing trades (1-5 days maximum)
   - Event-driven plays with clear catalysts

❌ WHEN TO AVOID TQQQ:

1. **High Volatility Sideways Markets**:
   - VIX above 30
   - Market corrections with frequent reversals
   - Uncertain economic conditions

2. **Long-term Holdings**:
   - NEVER for retirement accounts
   - NEVER for buy-and-hold strategies
   - Volatility decay destroys long-term returns

3. **Market Uncertainty**:
   - Fed policy transitions
   - Geopolitical crises
   - Earnings season volatility

📋 SPECIFIC TRADING STRATEGIES:

1. **Volatility-Based Strategy**:
   - Only trade when 30-day QQQ volatility < 25%
   - Exit immediately when volatility > 35%
   - Use VIX as early warning system

2. **Momentum Strategy**:
   - Enter on breakouts above key resistance
   - Use 2-3% stop losses (strict discipline)
   - Take profits on momentum exhaustion signals

3. **Time-Based Limits**:
   - Maximum holding period: 1-2 weeks
   - Never hold through FOMC meetings
   - Avoid during major earnings releases

4. **Position Sizing & Risk Management**:
   - Never exceed 5% of total portfolio
   - Size positions inversely to volatility
   - Use Kelly criterion for optimal sizing
   - Set both profit targets AND stop losses

⚠️ CRITICAL WARNINGS:
- Can lose 20-50% even when QQQ is flat
- Overnight risk is magnified 3x
- Tax implications of frequent trading
- Psychological stress of high volatility

🎯 BOTTOM LINE RECOMMENDATION:
TQQQ is a sophisticated TRADING tool for experienced traders with:
• Strong risk management discipline
• Short-term directional conviction
• Ability to monitor positions closely
• Understanding of volatility decay mechanics

It is NOT suitable for passive investors or long-term wealth building.
Use it tactically, not strategically.""")



## Files Required for Students:
- `data_loader.py` - Data loading utilities
- `qqq_tqqq_prices.csv` - Historical price data
- `qqq_tqqq_returns.csv` - Daily returns data

**Total: 100 points**